# Maximal Marginal Relevance (MMR) Retriever Implementation (Lecture 47)

Standard similarity retrieval frequently yields redundant candidate documents that repeat identical facts in varied phrasing, wasting the LLM context window.
**Maximal Marginal Relevance (MMR)** dynamically penalizes candidate documents that are highly similar to previously selected documents according to the optimization formula:

$$\text{MMR} = \arg\max_{d_i \in R \setminus S} \left[ \lambda \cdot \text{Sim}_1(d_i, q) - (1 - \lambda) \cdot \max_{d_j \in S} \text{Sim}_2(d_i, d_j) \right]$$

This notebook demonstrates:
1. Ingesting a redundant multi-document corpus into an in-memory FAISS index.
2. Directly comparing `search_type="similarity"` vs `search_type="mmr"`.
3. Evaluating the effect of the diversity parameter `lambda_mult` (from 0.1 to 1.0).
4. Quantifying inter-document redundancy via pairwise cosine distance.
5. Assembling an end-to-end LCEL RAG pipeline with `ChatGroq`.

## 1. Setup & Environment Initialization
Load dependencies and initialize dense embeddings.

In [1]:
import os
from typing import List, Dict, Any
import numpy as np
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings model initialized.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_19884\2580083125.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model initialized.


## 2. Ingesting Redundant Corpus
Construct a corpus with overlapping and complementary perspectives.

In [2]:
corpus_docs: List[Document] = [
    # Redundant group A: Python origins and readability
    Document(
        page_content="Python was conceived by Guido van Rossum in the late 1980s as a successor to the ABC language focusing on syntax readability.",
        metadata={"id": "doc_py_history_1", "topic": "history"}
    ),
    Document(
        page_content="Guido van Rossum developed Python in 1989 at CWI in the Netherlands to provide an intuitive, human-readable programming language.",
        metadata={"id": "doc_py_history_2", "topic": "history"}
    ),
    Document(
        page_content="The core philosophy of Python emphasizes code readability and uncluttered syntax using significant whitespace indentation.",
        metadata={"id": "doc_py_syntax_3", "topic": "history"}
    ),
    # Complementary group B: Data Science & AI ecosystem
    Document(
        page_content="Python is the dominant programming language in artificial intelligence, machine learning, and data science via PyTorch and NumPy.",
        metadata={"id": "doc_py_ai_4", "topic": "ai_data"}
    ),
    # Complementary group C: Web development & APIs
    Document(
        page_content="Modern web backends and high-performance microservices are built in Python using asynchronous frameworks like FastAPI and Django.",
        metadata={"id": "doc_py_web_5", "topic": "web_api"}
    ),
    # Complementary group D: Systems & Automation
    Document(
        page_content="DevOps engineers utilize Python extensively for infrastructure scripting, system automation, and cloud management tools.",
        metadata={"id": "doc_py_devops_6", "topic": "devops"}
    ),
]

vector_store = FAISS.from_documents(corpus_docs, embeddings)
print(f"FAISS index populated with {vector_store.index.ntotal} documents.")

FAISS index populated with 6 documents.


## 3. Side-by-Side Comparison: Similarity vs. MMR Retrieval
Execute an exploratory query and compare information diversity.

In [3]:
query = "What is Python and what are its key practical applications across industries?"

# Standard similarity retriever (k=3)
sim_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# MMR retriever (k=3, fetch_k=6, lambda_mult=0.5)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": 0.5}
)

sim_results = sim_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

print("=== Standard Similarity Search Results (Redundancy Trap) ===")
for rank, doc in enumerate(sim_results, start=1):
    print(f"Rank {rank} [{doc.metadata['id']} | Topic: {doc.metadata['topic']}]: {doc.page_content}")

print("\n=== Maximal Marginal Relevance (MMR) Results (Balanced Diversity) ===")
for rank, doc in enumerate(mmr_results, start=1):
    print(f"Rank {rank} [{doc.metadata['id']} | Topic: {doc.metadata['topic']}]: {doc.page_content}")

sim_topics = set(d.metadata["topic"] for d in sim_results)
mmr_topics = set(d.metadata["topic"] for d in mmr_results)

print(f"\nUnique topics covered - Similarity: {len(sim_topics)} | MMR: {len(mmr_topics)}")
assert len(mmr_topics) >= 2, "MMR must retrieve documents covering diverse topic facets"
print("Diversity comparison assertion passed.")

=== Standard Similarity Search Results (Redundancy Trap) ===
Rank 1 [doc_py_ai_4 | Topic: ai_data]: Python is the dominant programming language in artificial intelligence, machine learning, and data science via PyTorch and NumPy.
Rank 2 [doc_py_devops_6 | Topic: devops]: DevOps engineers utilize Python extensively for infrastructure scripting, system automation, and cloud management tools.
Rank 3 [doc_py_history_2 | Topic: history]: Guido van Rossum developed Python in 1989 at CWI in the Netherlands to provide an intuitive, human-readable programming language.

=== Maximal Marginal Relevance (MMR) Results (Balanced Diversity) ===
Rank 1 [doc_py_ai_4 | Topic: ai_data]: Python is the dominant programming language in artificial intelligence, machine learning, and data science via PyTorch and NumPy.
Rank 2 [doc_py_web_5 | Topic: web_api]: Modern web backends and high-performance microservices are built in Python using asynchronous frameworks like FastAPI and Django.
Rank 3 [doc_py_devops_6

## 4. Parameter Sensitivity Analysis (`lambda_mult` Sweep)
Observe the trade-off between pure relevance (lambda=1.0) and high diversity (lambda=0.1).

In [4]:
lambda_values = [0.1, 0.5, 0.9, 1.0]

print("Sensitivity Sweep for lambda_mult:")
for l_val in lambda_values:
    sweep_retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 6, "lambda_mult": l_val}
    )
    res = sweep_retriever.invoke(query)
    doc_ids = [d.metadata["id"] for d in res]
    topics = [d.metadata["topic"] for d in res]
    print(f"  lambda={l_val:.2f} -> IDs: {doc_ids} | Topics: {topics}")

assert len(res) == 3, "Sweep retriever must return exactly k=3 documents"
print("Sensitivity sweep passed.")

Sensitivity Sweep for lambda_mult:
  lambda=0.10 -> IDs: ['doc_py_ai_4', 'doc_py_web_5', 'doc_py_history_1'] | Topics: ['ai_data', 'web_api', 'history']
  lambda=0.50 -> IDs: ['doc_py_ai_4', 'doc_py_web_5', 'doc_py_devops_6'] | Topics: ['ai_data', 'web_api', 'devops']
  lambda=0.90 -> IDs: ['doc_py_ai_4', 'doc_py_devops_6', 'doc_py_web_5'] | Topics: ['ai_data', 'devops', 'web_api']
  lambda=1.00 -> IDs: ['doc_py_ai_4', 'doc_py_devops_6', 'doc_py_history_2'] | Topics: ['ai_data', 'devops', 'history']
Sensitivity sweep passed.


## 5. End-to-End LCEL RAG Pipeline Integration with ChatGroq
Feed diversified MMR context into a production generation chain.

In [5]:
def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(f"[{d.metadata['id']} ({d.metadata['topic']})]: {d.page_content}" for d in docs)

template = """You are an enterprise software architect. Provide a comprehensive summary based only on the context below:
Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

rag_chain = (
    {"context": mmr_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

final_response = rag_chain.invoke(query)
print("=== Final Augmented Answer with Diversified MMR Context ===\n")
print(final_response.strip())

assert len(final_response) > 50, "Generated answer must not be empty"
print("\nEnd-to-end MMR RAG pipeline assertion passed.")

=== Final Augmented Answer with Diversified MMR Context ===

**Python – a versatile, high‑level programming language**

Python is a general‑purpose, interpreted language known for its readability, extensive standard library, and a vibrant ecosystem of third‑party packages. Its design encourages rapid development and clear, maintainable code, making it a go‑to choice for many modern software projects.

---

### Key practical applications across industries

| Domain | What Python is used for | Representative libraries / frameworks |
|--------|------------------------|---------------------------------------|
| **Artificial Intelligence / Machine Learning / Data Science** | Building, training, and deploying predictive models; data wrangling, visualization, and statistical analysis | **PyTorch** (deep learning), **NumPy** (numerical computing), **pandas**, **scikit‑learn**, **TensorFlow** |
| **Web Development & Microservices** | Creating RESTful APIs, full‑stack web applications, and high‑